In [1]:
###!/usr/bin/env python
################################################
# New style 
# ###############################################
import sys

rootdir_ = '../'
if ( rootdir_ not in sys.path ):
    sys.path.append(rootdir_)
    print( f" a path to {rootdir_} added in {__name__} ")


from Utils import GridUtils as GrU
from Utils import MakePressures as MkP
from Utils import utils as uti
from Utils import MyConstants as Co
from Utils import time_utils as tuti
from Utils import numerical_utils as nuti

import analysis_utils as auti
import file_utils as futi
import event_utils as euti
import event_io as eio


#from PyRegridding.Utils import MakePressures as MkP
#from Drivers import RegridField as RgF
import RegridField as RgF

# The usual
from datetime import date
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# for smoothing , nonlienar colors ...
from scipy.ndimage import uniform_filter
from scipy.ndimage import gaussian_filter
import matplotlib.colors as mcolors

# Some other useful packages 
import copy
import time
import cftime
import yaml
import numbers
import pickle

# Some other useful packages 
import importlib
from pathlib import Path

import mlp_to_pptx


importlib.reload( auti )
importlib.reload( futi )
importlib.reload( euti )
importlib.reload( eio )

Rdair=Co.Rdair()


 a path to ../ added in __main__ 
 Utils.MyConstants in /glade/work/juliob/HiRes_ana_dev/Drivers/Utils 
Using Flexible parallel/serial VertRegrid 
 Utils.MyConstants in /glade/work/juliob/HiRes_ana_dev/Drivers/Utils 
 a path to /glade/work/juliob added in Utils.numerical_utils 


In [2]:
# This allow both dict.key and dict['key'] syntax
class AttrDict(dict):
    def __getattr__(self, key):
        try:
            return self[key]
        except KeyError:
            raise AttributeError(f"'AttrDict' object has no attribute '{key}'")

    def __setattr__(self, key, value):
        self[key] = value

    def __delattr__(self, key):
        try:
            del self[key]
        except KeyError:
            raise AttributeError(f"'AttrDict' object has no attribute '{key}'")



In [3]:
recalculate = False # False 
read_stored_El = not recalculate

print( f"Redoing A and Els ={recalculate}, Readin stored = {read_stored_El} ")


Redoing A and Els =False, Readin stored = True 


In [4]:
%%time
original_3x3_study=False
current_study=True

### Initialize to False
subsample_time_12_06_00 = False
transfer_x_ne240 = False
transfer_x_mpas  = False


if recalculate == False and read_stored_El==True:

    #fEl1 = '/glade/derecho/scratch/juliob/archive/GW_event_analysis/PKL/c153_topfix_ne240pg3_FMTHIST_xic_x03_2007-07-15-00000-x-2007-09-14-64800_60S-40S_ocean_EvZ10km_rho_epwp.pkl'
    #fEl2 = '/glade/derecho/scratch/juliob/archive/GW_event_analysis/PKL/c153_topfix_ne240pg3_FMTHIST_xic_x03_2007-07-15-00000-x-2007-09-14-64800_35N-70N_ocean_EvZ10km_rho_epwp.pkl'

    
    #fEl1 = '/glade/derecho/scratch/juliob/archive/GW_event_analysis/PKL/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_60S-40S_ocean_EvZ10km_rho_epwp.pkl'
    #fEl2 = '/glade/derecho/scratch/juliob/archive/GW_event_analysis/PKL/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_35N-70N_ocean_EvZ10km_rho_epwp.pkl'


    #fEl1 = '/glade/derecho/scratch/juliob/archive/GW_event_analysis/PKL/c124_dyamond1_prod2_2016-08-01-10800-x-2016-08-31-75600_60S-40S_ocean_EvZ10km_rho_epwp.pkl'
    #fEl2 = '/glade/derecho/scratch/juliob/archive/GW_event_analysis/PKL/c124_dyamond1_prod2_2016-08-01-10800-x-2016-08-31-75600_35N-70N_ocean_EvZ10km_rho_epwp.pkl'

    
    
    #fEl1 = '/glade/derecho/scratch/juliob/archive/GW_event_analysis/PKL/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_60S-40S_ocean_EvZ10km_rho_epwp_v2.pkl'
    #fEl2 = '/glade/derecho/scratch/juliob/archive/GW_event_analysis/PKL/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_35N-70N_ocean_EvZ10km_rho_epwp_v2.pkl'
    #fEl1 = '/glade/derecho/scratch/juliob/archive/GW_event_analysis/PKL/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_60S-40S_ocean_EvZ10km_rho_epwp_ftp5X5_v2.pkl'
    #fEl2 = '/glade/derecho/scratch/juliob/archive/GW_event_analysis/PKL/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_35N-70N_ocean_EvZ10km_rho_epwp_ftp5X5_v2.pkl'

    #fEl1 = '/glade/derecho/scratch/juliob/archive/GW_event_analysis/PKL/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_60S-40S_ocean_EvZ10km_rho_epwp_v2.pkl'
    #fEl2 = '/glade/derecho/scratch/juliob/archive/GW_event_analysis/PKL/c153_topfix_ne240pg3_FMTHIST_xic_x02_2004-07-15-00000-x-2004-09-14-64800_60S-40S_ocean_EvZ10km_rho_epwp_ftp3X3_v2.pkl'

    if current_study==True:
        bdir = f"/glade/derecho/scratch/juliob/archive/GW_event_analysis/PKL"

        fEl1 = f"{bdir}/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_60S-40S_ocean_EvZ10km_rho_epwp_ftp5X5_v2.pkl"
        fEl2 = f"{bdir}/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_35N-70N_ocean_EvZ10km_rho_epwp_ftp5X5_v2.pkl"

        fEl1 = f"{bdir}/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_60S-40S_ocean_EvZ10km_rho_epwp_subt_ftp5X5_v3.pkl"
        fEl2 = f"{bdir}/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_35N-70N_ocean_EvZ10km_rho_epwp_subt_ftp5X5_v3.pkl"

        #fEl1 = f"{bdir}/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_60S-40S_ocean_EvZ10km_rho_epwp_subt_ftp5X5_Frac100%_v4.pkl"
        #fEl2 = f"{bdir}/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_35N-70N_ocean_EvZ10km_rho_epwp_subt_ftp5X5_Frac100%_v4.pkl"

        subsample_time_12_06_00 = True
        transfer_x_ne240 = False
        transfer_x_mpas  = False
    elif original_3x3_study==True:
        fEl1 = '/glade/derecho/scratch/juliob/archive/GW_event_analysis/PKL/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_60S-40S_ocean_EvZ10km_rho_epwp_v2.pkl'
        fEl2 = '/glade/derecho/scratch/juliob/archive/GW_event_analysis/PKL/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_35N-70N_ocean_EvZ10km_rho_epwp_v2.pkl'
        subsample_time_12_06_00 = False
        transfer_x_ne240 = False
        transfer_x_mpas  = False
    else:     
        # "subgenres" if not original_3x3_study
        subsample_time_12_06_00 = True
        transfer_x_ne240 = False
        transfer_x_mpas  = False
        if transfer_x_mpas == True and transfer_x_ne240 == True:
            sys.exit( f"transfer_x_mpas == True and transfer_x_ne240 == True")
            
        if transfer_x_mpas==True:
            fEl1 = '/glade/derecho/scratch/juliob/archive/GW_event_analysis/PKL/c153_topfix_ne240pg3_FMTHIST_xic_x02_2004-07-15-00000-x-2004-09-14-64800_60S-40S_ocean_EvZ10km_rho_epwp_ftp5X5_v3.pkl'
            fEl2 = '/glade/derecho/scratch/juliob/archive/GW_event_analysis/PKL/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_60S-40S_ocean_EvZ10km_rho_epwp_subt_ftp5X5_v3.pkl'
        elif subsample_time_12_06_00 ==True:
            #fEl1 = '/glade/derecho/scratch/juliob/archive/GW_event_analysis/PKL/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_60S-40S_ocean_EvZ10km_rho_epwp_subt_ftp5X5_v3.pkl'
            #fEl2 = '/glade/derecho/scratch/juliob/archive/GW_event_analysis/PKL/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_35N-70N_ocean_EvZ10km_rho_epwp_subt_ftp5X5_v3.pkl'
            fEl1 = '/glade/derecho/scratch/juliob/archive/GW_event_analysis/PKL/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_70S-25S_ocean_EvZ10km_rho_epwp_subt_ftp5X5_v4.pkl'
            fEl2 = '/glade/derecho/scratch/juliob/archive/GW_event_analysis/PKL/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_25N-75N_ocean_EvZ10km_rho_epwp_subt_ftp5X5_v4.pkl'
            
            if transfer_x_ne240==True:
                fEl2 = '/glade/derecho/scratch/juliob/archive/GW_event_analysis/PKL/c153_topfix_ne240pg3_FMTHIST_xic_x02_2004-07-15-00000-x-2004-09-14-64800_60S-40S_ocean_EvZ10km_rho_epwp_ftp5X5_v3.pkl'
        else:
            fEl1 = '/glade/derecho/scratch/juliob/archive/GW_event_analysis/PKL/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_60S-40S_ocean_EvZ10km_rho_epwp_ftp5X5_v2.pkl'
            fEl2 = '/glade/derecho/scratch/juliob/archive/GW_event_analysis/PKL/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_35N-70N_ocean_EvZ10km_rho_epwp_ftp5X5_v2.pkl'
            
       
    
    print( f"Reading in stored = {read_stored_El} from \n {fEl1} ")
    with open(f'{fEl1}', 'rb') as f:
        El = pickle.load(f)
    
    print( f"Reading in stored = {read_stored_El} from \n {fEl2} ")
    with open(f'{fEl2}', 'rb') as f:
        El2 = pickle.load(f)


Reading in stored = True from 
 /glade/derecho/scratch/juliob/archive/GW_event_analysis/PKL/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_60S-40S_ocean_EvZ10km_rho_epwp_subt_ftp5X5_v3.pkl 
Reading in stored = True from 
 /glade/derecho/scratch/juliob/archive/GW_event_analysis/PKL/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_35N-70N_ocean_EvZ10km_rho_epwp_subt_ftp5X5_v3.pkl 
CPU times: user 13 ms, sys: 1min 31s, total: 1min 31s
Wall time: 3min 14s


In [5]:
##################
import importlib                                                                                                                                                                                                                                                                                              
import mlp_utils as mlu  
import predictors as Predi
importlib.reload(mlu) 
#importlib.reload(RF)
importlib.reload(Predi)
importlib.reload(eio)
importlib.reload( euti )

Eco  =   copy.deepcopy( El[0] )  #e
Eco2 =   copy.deepcopy( El2[0] ) #e

nv1, nt_v, nz_v, ny_v, nx_v = np.shape(Eco.u_4D)

extratag=""
vers_ = "_v4"

no_memory=False
time_average = True

if subsample_time_12_06_00 ==True:
    extratag=extratag+'_subt'

if transfer_x_ne240 ==True:
    extratag=extratag+'_Xne240'
if transfer_x_mpas ==True:
    extratag=extratag+'_Xmpas'

if time_average == True:
    Eco = euti.vtime_avg_events(Eco)
    Eco2 = euti.vtime_avg_events(Eco2)
    extratag=extratag+'_TimeAvg'

 In vtime_avg_events time4D - shape (18309,) 
 In vtime_avg_events lat4D - shape (18309,) 
 In vtime_avg_events lon4D - shape (18309,) 
 In vtime_avg_events u_4D - shape (18309, 3, 58, 11, 11) 
 In vtime_avg_events v_4D - shape (18309, 3, 58, 11, 11) 
 In vtime_avg_events htopo_4D - shape (18309, 3, 11, 11) 
 In vtime_avg_events zeta_4D - shape (18309, 3, 58, 11, 11) 
 In vtime_avg_events tilt_4D - shape (18309, 3, 58, 11, 11) 
 In vtime_avg_events fgf_4D - shape (18309, 3, 58, 11, 11) 
 In vtime_avg_events upwp_4D - shape (18309, 3, 58, 11, 11) 
 In vtime_avg_events vpwp_4D - shape (18309, 3, 58, 11, 11) 
 In vtime_avg_events epwp_4D - shape (18309, 3, 58, 11, 11) 
 In vtime_avg_events th_4D - shape (18309, 3, 58, 11, 11) 
 In vtime_avg_events stab_4D - shape (18309, 3, 58, 11, 11) 
 In vtime_avg_events precl_4D - shape (18309, 3, 58, 11, 11) 
 In vtime_avg_events thpwp_4D - shape (18309, 3, 58, 11, 11) 
 In vtime_avg_events time4D - shape (7000,) 
 In vtime_avg_events lat4D - shape (

In [6]:
Eco.upwp_4D.shape

(18309, 1, 58, 11, 11)

In [7]:
scale_precip = 86_400. * 1_000.
Eco.precl_4D = scale_precip * Eco.precl_4D
Eco2.precl_4D = scale_precip * Eco2.precl_4D
scale_precip = 1.0

prec_gt_3 = False
prec_lt_3 = False

if prec_gt_3 == True and prec_lt_3 == True:
    sys.exit()

print( f" Eco shape {np.shape( Eco.time4D )} " )
if prec_gt_3 == True:
    prook = scale_precip * np.mean( Eco.precl_4D , axis=(1,2,3,4) )
    voo = prook > 3. 
    Eco = euti.subselect_events(Eco=Eco, keep=voo, event_axis=0)
    extratag=extratag+'_precGT3'
elif prec_lt_3 == True:
    prook = scale_precip * np.mean( Eco.precl_4D , axis=(1,2,3,4) )
    voo = prook < 3. 
    Eco = euti.subselect_events(Eco=Eco, keep=voo, event_axis=0)
    extratag=extratag+'_precLT3'

print( f" Eco shape {np.shape( Eco.time4D )} ")

print( f" Eco2 shape {np.shape( Eco2.time4D )} ")
if prec_gt_3 == True:
    prook = scale_precip * np.mean( Eco2.precl_4D , axis=(1,2,3,4) )
    voo = prook > 3. 
    Eco2 = euti.subselect_events(Eco=Eco2, keep=voo, event_axis=0)
elif prec_lt_3 == True:
    prook = scale_precip * np.mean( Eco2.precl_4D , axis=(1,2,3,4) )
    voo = prook < 3. 
    Eco2 = euti.subselect_events(Eco=Eco2, keep=voo, event_axis=0)
print( f" Eco2 shape {np.shape( Eco2.time4D )} ")

print( f"Eco  precip min, max {np.min( scale_precip * np.mean( Eco.precl_4D , axis=(1,2,3,4) ) )} - {np.max( scale_precip * np.mean( Eco.precl_4D , axis=(1,2,3,4) ) )}" )
print( f"Eco2 precip min, max {np.min( scale_precip * np.mean( Eco2.precl_4D , axis=(1,2,3,4) ) )} - {np.max( scale_precip * np.mean( Eco2.precl_4D , axis=(1,2,3,4) ) )}" )



zlev =   El[0].zlevA
z0=np.argmin( np.abs( zlev-0.))
z3=np.argmin( np.abs( zlev-3000.))
z5=np.argmin( np.abs( zlev-5000.))
z6=np.argmin( np.abs( zlev-6000.))
z7=np.argmin( np.abs( zlev-7000.))
z10=np.argmin( np.abs( zlev-10000.))
z11=np.argmin( np.abs( zlev-11000.))
z12=np.argmin( np.abs( zlev-12000.))
z15=np.argmin( np.abs( zlev-15000.))



MyHyperparameters = Predi.reset_hyperparameters()
sweep_results = []   # accumulates across runs

do_sweep = [False, False, False, False, False ]
do_sweep[2] = True

if do_sweep[0]==True:

    predictor_sets = [
        ('baseline',        ['u_4D', 'v_4D', 'th_4D']),
        ('stab',            ['u_4D', 'v_4D', 'stab_4D']),
        ('stab+zeta',       ['u_4D', 'v_4D', 'stab_4D', 'zeta_4D']),
        ('stab+tilt',       ['u_4D', 'v_4D', 'stab_4D', 'tilt_4D']),
        ('stab+fgf',        ['u_4D', 'v_4D', 'stab_4D', 'fgf_4D']),
        ('stab+zeta+tilt',  ['u_4D', 'v_4D', 'stab_4D', 'zeta_4D', 'tilt_4D']),
        ('stab+zeta+fgf',   ['u_4D', 'v_4D', 'stab_4D', 'zeta_4D', 'fgf_4D']),
        ('precl+zeta+tilt', ['u_4D', 'v_4D', 'precl_4D', 'zeta_4D', 'tilt_4D']),
        ('precl+tilt+fgf',  ['u_4D', 'v_4D', 'precl_4D', 'tilt_4D', 'fgf_4D']),
        ('full(stab)',      ['u_4D', 'v_4D', 'stab_4D', 'zeta_4D', 'tilt_4D', 'fgf_4D']),
        ('full(precl)',     ['u_4D', 'v_4D', 'precl_4D', 'zeta_4D', 'tilt_4D', 'fgf_4D']),
    ]

    MyHyperparameters['predictorSet']    = 'original_default'
    MyHyperparameters['z_targ']          = z10
    MyHyperparameters['targ_scaling']    = 1.
    MyHyperparameters['yrange']          = [1, -1]
    MyHyperparameters['lr_patience']     = 3

    sweep_tag = 'sweep1'

elif do_sweep[1]==True:
    predictor_sets = [
        ('precl+tilt',       ['u_4D', 'v_4D', 'precl_4D', 'tilt_4D']),
        ('precl+zeta+tilt',  ['u_4D', 'v_4D', 'precl_4D', 'zeta_4D', 'tilt_4D']),
        ('precl+tilt',       ['u_4D', 'v_4D', 'precl_4D', 'tilt_4D']),
        ('precl+zeta+tilt',  ['u_4D', 'v_4D', 'precl_4D', 'zeta_4D', 'tilt_4D']),
    ]

    MyHyperparameters['predictorSet']    = 'original_default'
    MyHyperparameters['z_targ']          = z10
    MyHyperparameters['targ_scaling']    = 1.
    MyHyperparameters['yrange']          = [1, -1]
    MyHyperparameters['lr_patience']     = 3

    sweep_tag = 'sweep2'

elif do_sweep[2]==True:
    MyHyperparameters['predictorSet']    = 'Dynamic-steer-launch'

    """
    predictor_sets = [
        ('tilt_levs+tilt',             ['tilt_4D', 'tilt_4D']),
        ('tilt_levs+precl',            ['tilt_4D', 'precl_4D']),
        ('tilt_levs+tilt+precl',       ['tilt_4D', 'tilt_4D','precl_4D']),
        ('vmag_levs+tilt',             ['vmag_4D', 'tilt_4D',]),
        ('vmag_levs+precl',            ['vmag_4D', 'precl_4D']),
        ('vmag_levs+tilt+precl',       ['vmag_4D', 'tilt_4D','precl_4D']),
        ('zeta_levs+tilt',             ['abs_zeta_4D', 'tilt_4D',]),
        ('zeta_levs+precl',            ['abs_zeta_4D', 'precl_4D']),
        ('zeta_levs+tilt+precl',       ['abs_zeta_4D', 'tilt_4D','precl_4D']),
    ]
    """
    """
    predictor_sets = [
        ('zeta_levs+tilt',             ['abs_zeta_4D', 'tilt_4D',]),
        ('zeta_levs+precl',            ['abs_zeta_4D', 'precl_4D']),
        ('zeta_levs+tilt+precl',       ['abs_zeta_4D', 'tilt_4D','precl_4D']),
    ]
    """
    predictor_sets = [
        ('tilt_levs+tilt',             ['tilt_4D', 'tilt_4D']),
        ('tilt_levs+precl',            ['tilt_4D', 'precl_4D']),
        ('tilt_levs+tilt+precl',       ['tilt_4D', 'tilt_4D','precl_4D']),
        ('tilt_levs+fgf',              ['tilt_4D', 'fgf_4D']),
        ('tilt_levs+tilt+fgf',         ['tilt_4D', 'tilt_4D','fgf_4D']),
    ]

    #MyHyperparameters['use_predictors'] = ['tilt_4D','zeta_4D','tilt_4D'  , 'precl_4D' ] #'thpwp_4D']  # These are the best   
    MyHyperparameters['z_targ']          = -1 #z10
    MyHyperparameters['targ_scaling']    = 1.
    MyHyperparameters['yrange']          = [1,-1] #[5,6] #[1, -1]
    MyHyperparameters['xrange']          = None #[5,6] #[1, -1]
    MyHyperparameters['trange']          = None #[nt_v-1, nt_v]
    MyHyperparameters['hidden_dims']     = (64,64) # (64,64) #(64,64) # (32,32) #(16,16)  #(8,8)
    MyHyperparameters['lr_patience']     = 15

    sweep_tag = 'sweep3'

elif do_sweep[3]==True:

    predictor_sets = [
        ('baseline',               ['u_4D', 'v_4D', 'th_4D']),
        ('winds+precl',            ['u_4D', 'v_4D', 'precl_4D']),
        ('winds+zeta',             ['u_4D', 'v_4D', 'zeta_4D']),
        ('winds+tilt',             ['u_4D', 'v_4D', 'tilt_4D']),
        ('winds+precl+zeta',       ['u_4D', 'v_4D', 'precl_4D', 'zeta_4D']),
        ('winds+precl+tilt',       ['u_4D', 'v_4D', 'precl_4D', 'tilt_4D']),
        ('winds+zeta+tilt',        ['u_4D', 'v_4D', 'zeta_4D', 'tilt_4D']),
        ('winds+precl+tilt+zeta',  ['u_4D', 'v_4D', 'precl_4D', 'zeta_4D', 'tilt_4D']),
    ]

    MyHyperparameters['predictorSet']    = 'original_default'
    MyHyperparameters['z_targ']          = z10
    MyHyperparameters['targ_scaling']    = 1.
    MyHyperparameters['yrange']          = [1, -1]
    MyHyperparameters['lr_patience']     = 3

    sweep_tag = 'sweep4'

else: 
    print( This_cant_happen )    


if no_memory==True:
    MyHyperparameters['trange']  = [nt_v-1, nt_v]
    extratag = extratag +'_noMem'

extratag=f"{extratag}{vers_}"
# 'extratag' should be finished at this point. Don't modify further.
print( f"Final extratag {extratag} ")

for run_name, use_pred in predictor_sets:
    MyHyperparameters['use_predictors'] = use_pred
    
    
    predictors, predictor_names, use_predictors, key_z, short_desc, yv = \
        Predi.build_predictor_set(Eco, zlev, MyHyperparameters)
    
    mlp_model, mlp_results = mlu.fit_mlp_general(
        predictors      = predictors,
        predictor_names = predictor_names,
        target          = yv,
        event_times     = Eco.time4D,
        train_interval  = (48, 248),
        test_interval   = (0, 28),
        **{k: MyHyperparameters[k] for k in [
            'hidden_dims', 'dropout', 'batch_size', 'lr',
            'weight_decay', 'patience', 'max_epochs',
            'lr_patience', 'lr_factor', 'min_lr',
            'loss_power', 'log_predictor_patterns',
        ]}
    )
    meta = {
        'feat_scaler':   mlp_results['feat_scaler'],
        'target_scaler': mlp_results['target_scaler'],
        'log_predictor_patterns': mlp_results['log_predictor_patterns'],
        'log_pred_mask': mlp_results['log_pred_mask'],
    }
    device = 'cpu'  # or 'cuda' if you're on GPU

    ######### TRANSFER LEARNING ????????  #################################################
    from scipy import stats
    
    
    #Eco2 = El2[0]
    zlev2 = Eco2.zlevA
    
    predictors_2, predictor_names, use_predictors, key_z, short_desc, yv_2 = \
        Predi.build_predictor_set(Eco2, zlev2, MyHyperparameters)
    
    
    
    ################################
    
    y_pred_2 = mlu.apply_mlp(mlp_model, meta, device, predictors_2)
    
    
    print(y_pred_2.shape)
    print(yv_2.shape)
    
    
    
    eps = 1e-12
    ly_pred_2       = np.log(np.maximum(y_pred_2,  eps))
    ly_targ_2       = np.log(np.maximum(yv_2        ,  eps))
    r_log, _ = stats.pearsonr(ly_targ_2, ly_pred_2)
    print(r_log )

    sweep_results.append((
        run_name,
        mlp_results,
        {'y_pred': y_pred_2, 'y_targ': yv_2, 'label': 'NH transfer (El2)'},
        MyHyperparameters.copy()
        ))

 Eco shape (18309,) 
 Eco shape (18309,) 
 Eco2 shape (7000,) 
 Eco2 shape (7000,) 
Eco  precip min, max 1.3520024509727672e-05 - 25.49420079384304
Eco2 precip min, max 3.8110618551030674e-06 - 29.794010641255504
Final extratag _subt_TimeAvg_v4 
aded abs_zeta_4D ot Eco 
Dynamic steering and launch based on tilt_4D
 U_wv_src_mm t=0, dynamic  , shape = (18309,)  ,minmax=0.10836033557874226 - 34.73927003133501
 avg(tilt_4D) t=0, dynamic z (ZS-ZL avg)  , shape = (18309,) ,minmax=3.756970127329374e-08 - 1.3591480662488364e-06
 avg(tilt_4D) t=0, dynamic z (ZL)  , shape = (18309,) ,minmax=1.9226635599873028e-08 - 1.4180583217059225e-06
 avg(tilt_4D) t=0, dynamic z (ZS)  , shape = (18309,) ,minmax=2.4047327169965954e-08 - 1.2008708844467966e-06
 avg(tilt_4D) t=0, dynamic z (0-ZS avg)  , shape = (18309,) ,minmax=2.8976502551668643e-08 - 1.5892831483508373e-06
Created predictor set for dynamic steer and launch 
In build_target function 
trying dynamic launch level: shape = (18309, 1)
Predictor s

In [8]:
#stop_this_here  = "stop_this_here"
print( stop_this_here )

NameError: name 'stop_this_here' is not defined

In [9]:
print( f"How many hidden dims: {len(MyHyperparameters['hidden_dims'])}")
arch_ ='_mlp'
nlen = len(MyHyperparameters['hidden_dims'])
for i in np.arange(nlen):
    print( f"Dim[{i}] = {MyHyperparameters['hidden_dims'][i]}" )
    arch_ = f"{arch_}{MyHyperparameters['hidden_dims'][i]}x"

print( f"Architecture tag = {arch_}" )

if MyHyperparameters['yrange'] is not None:
    y0,y1 = MyHyperparameters['yrange']
    if y0 <= -1:
        y0 = y0+nx_v
    if y1 <= -1:
        y1 = y1+nx_v
    spac_ = f"_y{y0}-{y1-1}"
else:
    spac_ = f"_y%%"

if MyHyperparameters['xrange'] is not None:
    x0,x1 = MyHyperparameters['xrange']
    if x0 <= -1:
        x0 = x0+nx_v
    if x1 <= -1:
        x1 = x1+nx_v
    spac_ = f"{spac_}_x{x0}-{x1-1}"
else:
    spac_ = f"{spac_}_x%%"

print( f"Spatial window tag = {spac_}" )


How many hidden dims: 2
Dim[0] = 64
Dim[1] = 64
Architecture tag = _mlp64x64x
Spatial window tag = _y1-9_x%%


In [10]:
print(extratag)
extratag='_subt_TimeAvg_v3'

_subt_TimeAvg_v4


In [11]:
importlib.reload(mlp_to_pptx)


print( El[0].keys() )

footp_ = f"_ftp{El[0].peak_footprint[0]}X{El[0].peak_footprint[1]}"
pptx_file = f"/glade/work/juliob/HiRes_ana_dev/Drivers/Analysis/{Eco.case}_{sweep_tag}_results{footp_}{arch_}{spac_}{extratag}.pptx"
print( f"pptx => {pptx_file}" )

mlp_to_pptx.export_sweep_pptx(
    sweep_results = sweep_results,
    case_name     = Eco.case,
    pptx_out      = pptx_file,
)


dict_keys(['ds', 'fld', 'case', 'dycore', 'start_date', 'end_date', 'threshold', 'zlev_event', 'exclude_orography', 'subsample_time', 'lat_range', 'lon_range', 'N_events', 'Total_events', 'Frac_of_total_epwp', 'timeA', 'zlevA', 'latA', 'lonA', 'window_tyx', 'peak_footprint', 'time4D', 'lat4D', 'lon4D', 'u_4D', 'v_4D', 'htopo_4D', 'zeta_4D', 'tilt_4D', 'fgf_4D', 'upwp_4D', 'vpwp_4D', 'epwp_4D', 'th_4D', 'stab_4D', 'precl_4D', 'thpwp_4D'])
pptx => /glade/work/juliob/HiRes_ana_dev/Drivers/Analysis/cam77_dyamond1_prod1_sweep3_results_ftp5X5_mlp64x64x_y1-9_x%%_subt_TimeAvg_v3.pptx
mlp_to_pptx: generating sweep summary figure …
  [1/5] tilt_levs+tilt …
  [2/5] tilt_levs+precl …
  [3/5] tilt_levs+tilt+precl …
  [4/5] tilt_levs+fgf …
  [5/5] tilt_levs+tilt+fgf …
mlp_to_pptx: assembling sweep deck …
PPTX written -> /glade/work/juliob/HiRes_ana_dev/Drivers/Analysis/cam77_dyamond1_prod1_sweep3_results_ftp5X5_mlp64x64x_y1-9_x%%_subt_TimeAvg_v3.pptx
mlp_to_pptx: done  →  /glade/work/juliob/HiRes_an

In [ ]:
El[5].upwp_4D.shape

In [ ]:
v=10 # 1
fig,axs=plt.subplots( 1, 2, figsize=(17,7) )
ax=axs[0]
ax.plot( El[5].upwp_4D[v,-1,:,:,:].mean(axis=(1,2) ), zlev )
ax.plot( El[5].vpwp_4D[v,-1,:,:,:].mean(axis=(1,2) ), zlev )
ax=axs[1]
#ax.plot( El[5].u_4D[v,-1,:,5,5], zlev )
#ax.plot( El[5].v_4D[v,-1,:,5,5], zlev )
ax.plot( El[5].u_4D[v,-1,:,:,:].mean(axis=(1,2) ), zlev )
ax.plot( El[5].v_4D[v,-1,:,:,:].mean(axis=(1,2) ), zlev )
#plt.plot( El[5].epwp_4D[v,-1,:,5,5], zlev )



In [ ]:
importlib.reload( euti )
eeoo = euti.vtime_avg_events(Eco)

In [ ]:
print( eeoo.keys() )
print( np.shape( eeoo.time4D) )

In [ ]:
import mlp_to_pdf
importlib.reload(mlp_to_pdf)

mlp_to_pdf.export_sweep_pdf(
    sweep_results = sweep_results,
    case_name     = Eco.case,
    pdf_out      = f"/glade/work/juliob/HiRes_ana_dev/Drivers/Analysis/{Eco.case}_{sweep_tag}_results{footp_}_v2.pdf",
)


In [ ]:
prec = np.mean( Eco.precl_4D[:,:,57,:,:], axis=( 2,3) )
prec.shape


plt.scatter( Eco.lon4D, Eco.lat4D, c=1000.*86_400.*prec[:,-1], cmap='Blues' )
plt.xlim(0,360)
plt.ylim(-90,90)
plt.colorbar()


In [ ]:
h=np.histogram( 1000.*86_400* prec.flatten(), bins=np.linspace(0., 25., num=2500) ) 
plt.plot( h[1][1:],h[0],'.')
print( h[1][0:4] )
print( h[0][0:4] )


In [ ]:
importlib.reload( auti )
auti.plot_xavg_compos( fld='zeta_4D', El=El )
auti.plot_xavg_compos( fld='tilt_4D', El=El )
auti.plot_xavg_compos( fld='fgf_4D', El=El )

In [ ]:
importlib.reload( auti )
auti.plot_xavg_compos( fld='zeta_4D', El=El2 )
auti.plot_xavg_compos( fld='tilt_4D', El=El2 )
auti.plot_xavg_compos( fld='fgf_4D', El=El2)

In [ ]:
mlu.plot_distributions(mlp_results, desc=short_desc  ) 

mlu.plot_mlp_results(mlp_results, top_n=20 ,desc=short_desc[0:200])

In [ ]:
#import random_forest as RF

Eco=   El[0]  #euti.combine_event_dicts(El[3], El[1], label_key='event_strength')
zlev=El[0].zlevA
z0=np.argmin( np.abs( zlev-0.))
z3=np.argmin( np.abs( zlev-3000.))
z5=np.argmin( np.abs( zlev-5000.))
z6=np.argmin( np.abs( zlev-6000.))
z7=np.argmin( np.abs( zlev-7000.))
z10=np.argmin( np.abs( zlev-10000.))
z11=np.argmin( np.abs( zlev-11000.))
z12=np.argmin( np.abs( zlev-12000.))
z15=np.argmin( np.abs( zlev-15000.))


print( np.shape(Eco.u_4D ))
dt = Eco.timeA[1] - Eco.timeA[0] 
print(dt.total_seconds())

In [ ]:
import predictors as Predi
importlib.reload( Predi )

MyHyperparameters = Predi.reset_hyperparameters()



In [ ]:
Eco=El[0]

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
matplotlib.use('module://matplotlib_inline.backend_inline')

In [ ]:

meta = {
    'feat_scaler':   mlp_results['feat_scaler'],
    'target_scaler': mlp_results['target_scaler'],
    'log_pred_mask': mlp_results['log_pred_mask'],
}
device = 'cpu'  # or 'cuda' if you're on GPU

In [ ]:
######### TRANSFER LEARNING ????????  #################################################
from scipy import stats


Eco2 = El2[0]

predictors_2, predictor_names, use_predictors, key_z, short_desc, yv_2 = \
    Predi.build_predictor_set(Eco2, zlev, MyHyperparameters)



################################

y_pred_2 = mlu.apply_mlp(mlp_model, meta, device, predictors_2)


print(y_pred_2.shape)
print(yv_2.shape)



eps = 1e-12
ly_pred_2       = np.log(np.maximum(y_pred_2,  eps))
ly_targ_2       = np.log(np.maximum(yv_2        ,  eps))
r_log, _ = stats.pearsonr(ly_targ_2, ly_pred_2)
print(r_log )

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(6, 5))

# --- panel 1: log-log scatter ---
ax = axes
ax.scatter(ly_targ_2, ly_pred_2, alpha=0.3, s=10, color='steelblue')
lims = [min(ly_targ_2.min(), ly_pred_2.min()),
        max(ly_targ_2.max(), ly_pred_2.max())]
ax.plot(lims, lims, 'r--', lw=1)
r_log, _ = stats.pearsonr(ly_targ_2, ly_pred_2)
ax.set_xlabel('log(actual)')
ax.set_ylabel('log(predicted)')
ax.set_title(f'Log-space scatter  r={r_log:.3f}')

ax.text(0.5, -0.02, short_desc[0:180], ha='center', va='top', fontsize=9,
         transform=fig.transFigure)



In [ ]:
#########################################
import mlp_to_pptx
importlib.reload(mlp_to_pptx)
case_name='A_3rd_test'
mlp_to_pptx.export_mlp_pptx(
    mlp_results     = mlp_results,
    hyperparameters = MyHyperparameters,
    case_name       = case_name,
    pptx_out        = f"/glade/work/juliob/HiRes_ana_dev/Drivers/Analysis/{case_name}.pptx",
    transfer_results = {
        'y_pred' : y_pred_2,
        'y_targ' : yv_2,
        'label'  : 'NH transfer (El2)',
    },
)

In [ ]:
print( El[0].threshold )
print( np.log( El[0].threshold ))


In [ ]:
print( stop_here )

In [ ]:



El[bb].tilt_4D.shape

In [ ]:
fig,axs=plt.subplots( 1 , 4, figsize=(21,5)  )

bb=5
p=0
ax=axs[p]
ax.plot( np.mean( El[bb].tilt_4D, axis=(0,1,3,4) ) , zlev )
ax.plot( np.mean( El2[bb].tilt_4D, axis=(0,1,3,4) ) , zlev )
ax.set_ylim( 0,20_000 )
p=p+1
ax=axs[p]
ax.plot( np.mean( El[bb].zeta_4D, axis=(0,1,3,4) ) , zlev )
ax.plot( np.mean( El2[bb].zeta_4D, axis=(0,1,3,4) ) , zlev )
ax.set_ylim( 0,20_000 )
p=p+1
ax=axs[p]
ax.plot( np.mean( El[bb].u_4D, axis=(0,1,3,4) ) , zlev, color='blue' )
ax.plot( np.mean( El[bb].v_4D, axis=(0,1,3,4) ) , zlev, linestyle='--',color='blue' )

ax.plot( np.mean( El2[bb].u_4D, axis=(0,1,3,4) ) , zlev, color='orange' )
ax.plot( np.mean( El2[bb].v_4D, axis=(0,1,3,4) ) , zlev, linestyle='--',color='orange' )
ax.set_ylim( 0,20_000 )
p=p+1
ax=axs[p]
ax.plot( np.mean( El[bb].th_4D, axis=(0,1,3,4) ) , zlev )
ax.plot( np.mean( El2[bb].th_4D, axis=(0,1,3,4) ) , zlev )
ax.set_ylim( 0,20_000 )


In [ ]:
fig,axs=plt.subplots( 1 , 4, figsize=(21,5)  )

bb=0
p=0
ax=axs[p]
ax.plot( np.mean( El[bb].tilt_4D, axis=(0,1,3,4) ) , zlev )
ax.plot( np.mean( El2[bb].tilt_4D, axis=(0,1,3,4) ) , zlev )
ax.set_ylim( 0,20_000 )
p=p+1
ax=axs[p]
ax.plot( np.mean( El[bb].zeta_4D, axis=(0,1,3,4) ) , zlev )
ax.plot( np.mean( El2[bb].zeta_4D, axis=(0,1,3,4) ) , zlev )
ax.set_ylim( 0,20_000 )
p=p+1
ax=axs[p]
ax.plot( np.mean( El[bb].u_4D, axis=(0,1,3,4) ) , zlev, color='blue' )
ax.plot( np.mean( El[bb].v_4D, axis=(0,1,3,4) ) , zlev, linestyle='--',color='blue' )

ax.plot( np.mean( El2[bb].u_4D, axis=(0,1,3,4) ) , zlev, color='orange' )
ax.plot( np.mean( El2[bb].v_4D, axis=(0,1,3,4) ) , zlev, linestyle='--',color='orange' )
ax.set_ylim( 0,20_000 )
p=p+1
ax=axs[p]
ax.plot( np.mean( El[bb].th_4D, axis=(0,1,3,4) ) , zlev )
ax.plot( np.mean( El2[bb].th_4D, axis=(0,1,3,4) ) , zlev )
ax.set_ylim( 0,20_000 )


In [ ]:
plev = 100_000.* np.exp( -zlev / 7_000. )
zetooo= np.abs(np.mean( El[bb].zeta_4D, axis=(0,1,3,4) ))**2

p_steer = np.sum( zetooo*plev ) /np.sum( zetooo) 
z_steer = -7_000.*np.log( p_steer/100_000. )
print( f"{p_steer} Pa/ {z_steer} m")


In [ ]:
importlib.reload( auti )
auti.plot_xavg_compos( fld='zeta_4D', El=El )
auti.plot_xavg_compos( fld='tilt_4D', El=El )
auti.plot_xavg_compos( fld='fgf_4D', El=El )


In [ ]:
print(f"{El[5].threshold[0]:.4f}-{El[5].threshold[1]:.4g}".replace('.','_').replace('+','') )

In [ ]:
importlib.reload(eio)

eio.write_event_ds(El=El)